Experiments for the LSTM Neural Network Model
---
Performs grid search on __ different combinations to determine the best NN model for dementia classifcation.

Grid search is run on all three transcript types to determine the best model for each PFT, CTD, and SFT.

In [1]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer

sys.path.append(str(Path("..").resolve()))

DATA_PATH = Path("..") / "data" / "transcripts_cleaned.csv"

transcript_df = pd.read_csv(DATA_PATH)

print(transcript_df.shape)
transcript_df.head()

(157, 6)


,Record-ID,Class,Transcript_PFT,Transcript_CTD,Transcript_SFT,Label
0,Process-rec-001,MCI,"people, partner, plate, platter, pants, porter...",NaN,"<pause_medium> giraffe, kangaroo, lion, tiger,...",1
1,Process-rec-002,MCI,"<pause_short> pipe, plane, people <pause_mediu...",<pause_medium> there’s a lad stood on the stoo...,"<pause_short> dogs, cats, birds <pause_short> ...",1
2,Process-rec-003,MCI,"um <pause_short> purple, pale, placid <pause_s...","<pause_medium> um, the picture is of a kitchen...","cow, bull, ewe, ram, chicken, goose, um <sigh>...",1
3,Process-rec-004,MCI,plank <pause_short> pool <pause_short> swimmin...,"a mother presumably, or a fe, an adult female ...",um <pause_short> impala <pause_short> er cheet...,1
4,Process-rec-005,MCI,"it’s er pillock, er post box, er pyracanthas, ...","‘50s style er scene of domestic um confusion, ...","dog, cat, giraffe, wallaby, kangaroo, tortoise...",1


In [2]:
TRANSCRIPT_COLS = ["Transcript_PFT", "Transcript_CTD", "Transcript_SFT"]

print("NaN counts per transcript type: ")
for col in TRANSCRIPT_COLS:
    n_nans = transcript_df[col].isna().sum()
    print(f"{col}: {n_nans} NaNs")

NaN counts per transcript type: 
Transcript_PFT: 0 NaNs
Transcript_CTD: 1 NaNs
Transcript_SFT: 5 NaNs


In [3]:
from transcript_preprocessing import load_transcript_splits
df_by_transcript = load_transcript_splits(DATA_PATH)

for col in TRANSCRIPT_COLS:
    print(f"After removing NaNs for {col}: {len(df_by_transcript[col])} samples")

for col, df_clean in df_by_transcript.items():
    print(f"\nLabel counts for {col}:")
    print(df_clean["Class"].value_counts())


After removing NaNs for Transcript_PFT: 157 samples
After removing NaNs for Transcript_CTD: 156 samples
After removing NaNs for Transcript_SFT: 152 samples

Label counts for Transcript_PFT:
Class
HC          82
MCI         59
Dementia    16
Name: count, dtype: int64

Label counts for Transcript_CTD:
Class
HC          82
MCI         58
Dementia    16
Name: count, dtype: int64

Label counts for Transcript_SFT:
Class
HC          78
MCI         58
Dementia    16
Name: count, dtype: int64


In [4]:
# Analyze transcript lengths and vocab sizes to determine how to set max lengths anf if voacb limiting is necessary
length_stats = {}

for col in TRANSCRIPT_COLS:
    print(f"\n{col}:")
    
    df = df_by_transcript[col]
    texts = df[col].astype(str).tolist()
    
    # Fit tokenizer for analysis
    tokenizer = Tokenizer(oov_token="<UNK>")
    tokenizer.fit_on_texts(texts)
    
    vocab_size = len(tokenizer.word_index)
    
    # Convert to sequences
    sequences = tokenizer.texts_to_sequences(texts)
    lengths = np.array([len(seq) for seq in sequences])
    
    length_stats[col] = {
        "lengths": lengths,
        "tokenizer": tokenizer,
        "vocab_size": vocab_size,
    }
    
    print(f"Number of samples: {len(lengths)}")
    print(f"Vocab size (unique tokens): {vocab_size}")
    print(f"Min length:       {lengths.min()}")
    print(f"Max length:       {lengths.max()}")
    print(f"Mean length:      {lengths.mean():.2f}")



Transcript_PFT:
Number of samples: 157
Vocab size (unique tokens): 1267
Min length:       13
Max length:       94
Mean length:      47.58

Transcript_CTD:
Number of samples: 156
Vocab size (unique tokens): 1400
Min length:       13
Max length:       474
Mean length:      163.89

Transcript_SFT:
Number of samples: 152
Vocab size (unique tokens): 931
Min length:       25
Max length:       121
Mean length:      62.91


In [5]:
MAX_LEN_PFT = 94
MAX_LEN_CTD = 474
MAX_LEN_SFT = 121

MAX_LEN_BY_TRANSCRIPT = {
    "Transcript_PFT": MAX_LEN_PFT,
    "Transcript_CTD": MAX_LEN_CTD,
    "Transcript_SFT": MAX_LEN_SFT,
}

In [6]:
from transcript_preprocessing import get_stratified_kfold_splits
from utils import prepare_fold_data

# Verify padding/tokenization for each transcript type (for first fold only)
for transcript_col in TRANSCRIPT_COLS:
    print(f"\n{transcript_col}:")
    
    df = df_by_transcript[transcript_col]
    max_len = MAX_LEN_BY_TRANSCRIPT[transcript_col]
    
    y = df["Label"].values
    
    # Run Stratified K-Fold, and inspect the first fold
    for fold_idx, train_idx, val_idx in get_stratified_kfold_splits(
        transcript_df=df,
        transcript_col=transcript_col,
        label_col="Label",
        n_splits=5,
        seed=42,
    ):
        print(f"Fold {fold_idx}")
        
        X_train, y_train, X_val, y_val, tokenizer, vocab_size = prepare_fold_data(
            df=df,
            transcript_col=transcript_col,
            label_col="Label",
            train_idx=train_idx,
            val_idx=val_idx,
            max_len=max_len,
        )
        
        print("  X_train shape:", X_train.shape)
        print("  X_val shape:  ", X_val.shape)
        print("  y_train shape:", y_train.shape)
        print("  y_val shape:  ", y_val.shape)
        print("  max_len used: ", max_len)
        print("  vocab_size:   ", vocab_size)
        break



Transcript_PFT:
Fold 1
  X_train shape: (125, 94)
  X_val shape:   (32, 94)
  y_train shape: (125,)
  y_val shape:   (32,)
  max_len used:  94
  vocab_size:    1125

Transcript_CTD:
Fold 1
  X_train shape: (124, 474)
  X_val shape:   (32, 474)
  y_train shape: (124,)
  y_val shape:   (32,)
  max_len used:  474
  vocab_size:    1298

Transcript_SFT:
Fold 1
  X_train shape: (121, 121)
  X_val shape:   (31, 121)
  y_train shape: (121,)
  y_val shape:   (31,)
  max_len used:  121
  vocab_size:    873


In [7]:
import itertools

# Hyperparameter options for grid search
MODEL_TYPES = ["lstm", "bilstm"]
EMBEDDING_DIMS = [100, 200]
LSTM_UNITS = [64, 128]
DROPOUT_RATES = [0.3]
RECURRENT_DROPOUT_RATES = [0.0]
DENSE_UNITS_OPTIONS = [0, 64]  # 0 = no dense layer, 64 = add dense hidden layer
LEARNING_RATES = [1e-3]

def make_config_grid():
    """Create a list of config dicts for grid search."""
    grid = []
    for (
        model_type,
        emb_dim,
        lstm_units,
        dropout,
        rec_dropout,
        dense_units,
        lr,
    ) in itertools.product(
        MODEL_TYPES,
        EMBEDDING_DIMS,
        LSTM_UNITS,
        DROPOUT_RATES,
        RECURRENT_DROPOUT_RATES,
        DENSE_UNITS_OPTIONS,
        LEARNING_RATES,
    ):
        config = {
            "model_type": model_type,
            "embedding_dim": emb_dim,
            "lstm_units": lstm_units,
            "dropout": dropout,
            "recurrent_dropout": rec_dropout,
            "dense_units": dense_units,
            "learning_rate": lr,
        }
        grid.append(config)
    return grid

config_grid = make_config_grid()
print(f"Total configs: {len(config_grid)}")

Total configs: 16


In [8]:
from utils import run_cv_for_config
# Suppress TensorFlow warnings
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

all_results = []

for transcript_col in TRANSCRIPT_COLS:
    df = df_by_transcript[transcript_col]
    max_len = MAX_LEN_BY_TRANSCRIPT[transcript_col]

    print(f"\nGrid search for {transcript_col}:")
    print(f"Num samples: {len(df)}, max_len: {max_len}")

    for i, config in enumerate(config_grid, start=1):
        print(f"\n[{transcript_col}] Config {i}: {config}")

        summary = run_cv_for_config(
            transcript_col=transcript_col,
            df=df,
            config=config,
            max_len=max_len,
            n_splits=5,
            seed=42,
            batch_size=16,
            max_epochs=30,
        )

        # Merge config + metrics into one flat dict
        result_row = {
            "transcript_col": transcript_col,
            **config,
            **summary,
        }
        all_results.append(result_row)

# Convert to DataFrame and save results
results_df = pd.DataFrame(all_results)
results_df.to_csv("lstm_bilstm_grid_search_results.csv", index=False)

# See top configs per transcript type sorted by mean_macro_f1
display(
    results_df.sort_values(
        ["transcript_col", "mean_macro_f1"],
        ascending=[True, False],
    ).groupby("transcript_col").head(5)
)


Grid search for Transcript_PFT:
Num samples: 157, max_len: 94

[Transcript_PFT] Config 1: {'model_type': 'lstm', 'embedding_dim': 100, 'lstm_units': 64, 'dropout': 0.3, 'recurrent_dropout': 0.0, 'dense_units': 0, 'learning_rate': 0.001}

[Transcript_PFT] Config 2: {'model_type': 'lstm', 'embedding_dim': 100, 'lstm_units': 64, 'dropout': 0.3, 'recurrent_dropout': 0.0, 'dense_units': 64, 'learning_rate': 0.001}

[Transcript_PFT] Config 3: {'model_type': 'lstm', 'embedding_dim': 100, 'lstm_units': 128, 'dropout': 0.3, 'recurrent_dropout': 0.0, 'dense_units': 0, 'learning_rate': 0.001}

[Transcript_PFT] Config 4: {'model_type': 'lstm', 'embedding_dim': 100, 'lstm_units': 128, 'dropout': 0.3, 'recurrent_dropout': 0.0, 'dense_units': 64, 'learning_rate': 0.001}

[Transcript_PFT] Config 5: {'model_type': 'lstm', 'embedding_dim': 200, 'lstm_units': 64, 'dropout': 0.3, 'recurrent_dropout': 0.0, 'dense_units': 0, 'learning_rate': 0.001}

[Transcript_PFT] Config 6: {'model_type': 'lstm', 'embedd

,transcript_col,model_type,embedding_dim,lstm_units,dropout,recurrent_dropout,dense_units,learning_rate,mean_accuracy,std_accuracy,...,mean_weighted_f1,std_weighted_f1,mean_precision_macro,std_precision_macro,mean_recall_macro,std_recall_macro,mean_precision_weighted,std_precision_weighted,mean_recall_weighted,std_recall_weighted
23,Transcript_CTD,lstm,200,128,0.3,0.0,64,0.001,0.538306,0.067207,...,0.501930,0.082027,0.449468,0.113635,0.469712,0.136516,0.544389,0.093022,0.538306,0.067207
30,Transcript_CTD,bilstm,200,128,0.3,0.0,0,0.001,0.583669,0.112025,...,0.523379,0.139093,0.405270,0.218830,0.419660,0.123431,0.503625,0.172632,0.583669,0.112025
26,Transcript_CTD,bilstm,100,128,0.3,0.0,0,0.001,0.538105,0.104498,...,0.477721,0.130344,0.335982,0.144975,0.399666,0.103791,0.446767,0.153171,0.538105,0.104498
29,Transcript_CTD,bilstm,200,64,0.3,0.0,64,0.001,0.519153,0.089135,...,0.468764,0.106060,0.349005,0.108778,0.405110,0.087936,0.458502,0.121836,0.519153,0.089135
21,Transcript_CTD,lstm,200,64,0.3,0.0,64,0.001,0.475000,0.059503,...,0.446336,0.035877,0.371202,0.079579,0.356952,0.038872,0.470670,0.046703,0.475000,0.059503
6,Transcript_PFT,lstm,200,128,0.3,0.0,0,0.001,0.458266,0.072237,...,0.416584,0.043669,0.411177,0.103614,0.386794,0.067038,0.450939,0.046676,0.458266,0.072237
11,Transcript_PFT,bilstm,100,128,0.3,0.0,64,0.001,0.510484,0.064079,...,0.452313,0.071244,0.357213,0.103410,0.383296,0.038711,0.455366,0.102014,0.510484,0.064079
8,Transcript_PFT,bilstm,100,64,0.3,0.0,0,0.001,0.465121,0.027360,...,0.413066,0.042970,0.330348,0.097672,0.370469,0.072707,0.413297,0.081552,0.465121,0.027360
14,Transcript_PFT,bilstm,200,128,0.3,0.0,0,0.001,0.439315,0.132431,...,0.396899,0.093112,0.430244,0.146947,0.353691,0.065337,0.490799,0.121223,0.439315,0.132431
4,Transcript_PFT,lstm,200,64,0.3,0.0,0,0.001,0.470363,0.109128,...,0.407849,0.128334,0.359553,0.198786,0.356736,0.089584,0.418601,0.167271,0.470363,0.109128
